In this project I will try to create machine learning model to predict if price of sp&500 will go up or down

In [ ]:
#Wczytujemy potrzebne biblioteki
import pandas as pd
import numpy as np  
import yfinance as yf
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import precision_score

Poni?ej wczytujemy dane z serwisu yahoo finance
Robimy to za pomoc? biblioteki yfinance

In [ ]:
sp500 = yf.Ticker('^GSPC')

sp500 = sp500.history(period='max')

In [ ]:
sp500.plot.line(y='Close', title='S&P 500', ylabel='Price', xlabel='Date', use_index=True, figsize=(10, 5))

In [ ]:
# remove Dividends and Stock Splits columns
del sp500['Dividends']
del sp500['Stock Splits']   

In [ ]:
#Now we are going to create new column with tomorrow's price
sp500['Tomorrow'] = sp500['Close'].shift(-1)

In [ ]:
sp500['Target'] = (sp500['Tomorrow'] > sp500['Close']).astype(int)

In [ ]:
#Old prices can be problematic for our model so we are going to take data since 1990
sp500 = sp500.loc['1990-01-01':].copy()

Now we are starting building and training our model

In [ ]:
model = RandomForestClassifier(n_estimators=60, min_samples_split= 55, random_state=1)

In [ ]:
train = sp500.iloc[:-100] #PICKING ALL DATA EXCEPT LAST 100 DAYS FOR TRAINING
test = sp500.iloc[-100:] #PICKING LAST 100 DAYS FOR TESTING

predictors = ["Close", "Volume", "Open", "High", "Low"]
model.fit(train[predictors], train['Target'])

In [ ]:
preds = model.predict(test[predictors])

preds = pd.Series(preds, index=test.index)  

In [ ]:
precision_score(test['Target'], preds)

In [ ]:
# 0 = our predictions
combined = pd.concat([test['Target'], preds], axis=1)

combined.plot()

BACKTESTING
creating prediction function and backtest
Backtest is sth that will evaluate our model over time

In [ ]:
def predict(train, test, predictors, model):
    model.fit(train[predictors], train['Target'])
    preds = model.predict(test[predictors])
    preds = pd.Series(preds, index=test.index, name='Predictions')
    combined = pd.concat([test['Target'], preds], axis=1)
    return combined

In [ ]:
def backtest(data, model, predictors, start=2500, step=250):
    all_predictions = []
    for i in range(start, data.shape[0], step):
        train = data.iloc[0:i].copy()
        test = data.iloc[i:(i+step)].copy()
        predictions = predict(train, test, predictors, model)
        all_predictions.append(predictions)
    return pd.concat(all_predictions)

In [ ]:
predictions = backtest(sp500, model, predictors)

In [ ]:
predictions['Predictions'].value_counts()


In [ ]:
precision_score(predictions['Target'], predictions['Predictions']) 

In [ ]:
predictions['Target'].value_counts() / predictions.shape[0]

Prediction score naszego modelu wynosi 53%

